# 02 — Depuración Silver

Objetivo: recibir los parquets de Bronze, aplicar reglas de calidad, y guardar parquets limpios en Silver.

El propósito de este notebook es aplicar reglas de calidad, no explorar: cada decisión tiene una justificación documentada.

## IMPORTS Y CONFIGURACIÓN DE RUTAS

In [1]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib as plt


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


DATA_DIR = PROJECT_ROOT / 'data'
SILVER = DATA_DIR / 'silver'
SILVER.mkdir(parents=True, exist_ok=True)
BRONZE_SNAP = DATA_DIR / 'bronze' / 'snapshots'


print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')

Project root: /Volumes/SSDExterno/Lauris/Master/TFM/TFM-Hosteleria-AI
Data directory: /Volumes/SSDExterno/Lauris/Master/TFM/TFM-Hosteleria-AI/data


## SECCIÓN 0 — Setup

Cargamos los 12 ficheros parquet

In [2]:
df_tickets     = pd.read_parquet(BRONZE_SNAP / 'tickets_raw.parquet')
df_ventas      = pd.read_parquet(BRONZE_SNAP / 'ventas_raw.parquet')
df_reservas    = pd.read_parquet(BRONZE_SNAP / 'reservas_raw.parquet')
df_tips        = pd.read_parquet(BRONZE_SNAP / 'tips_raw.parquet')
df_artic       = pd.read_parquet(BRONZE_SNAP / 'articulos_raw.parquet')
df_depart      = pd.read_parquet(BRONZE_SNAP / 'departamentos_raw.parquet')
df_menu        = pd.read_parquet(BRONZE_SNAP / 'menu_raw.parquet')
df_festivos    = pd.read_parquet(BRONZE_SNAP / 'festivos_raw.parquet')
df_eventos     = pd.read_parquet(BRONZE_SNAP / 'eventos_raw.parquet')
meteo_diaria   = pd.read_parquet(BRONZE_SNAP / 'meteo_diaria_raw.parquet')
meteo_horaria  = pd.read_parquet(BRONZE_SNAP / 'meteo_horaria_raw.parquet')
total_articles = pd.read_parquet(BRONZE_SNAP / 'total_articles_raw.parquet')

## SECCIÓN 1 — Tabla de auditoría inicial

Antes de aplicar ninguna transformación, inspeccionamos el estado de los 12 datasets
tal como llegaron de Bronze. 

El objetivo es tener una fotografía del punto de partida
que sirva como referencia para documentar qué se modificó y por qué en las secciones
siguientes.

Para cada dataset se calculan cinco métricas:
- **filas / columnas**: dimensiones del dataset
- **nulos_pct_max**: porcentaje de nulos de la columna más afectada
- **duplicados_exactos**: filas idénticas en todas las columnas
- **fecha_min / fecha_max**: rango temporal de la columna de fecha operativa principal
  (NaT para datasets sin dimensión temporal propia: catálogos, tips, total_articles)python

In [8]:
def audit_dataset(nombre, df, col_fecha=None):
    """
    Genera un resumen de calidad de un DataFrame.

    Parameters
    ----------
    nombre : str
        Nombre identificador del dataset.
    df : pd.DataFrame
        Dataset a auditar.
    col_fecha : str or None
        Columna de fecha operativa principal. Si es None, fecha_min y fecha_max
        se devuelven como NaT.

    Returns
    -------
    dict
        Métricas de calidad del dataset.
    """
    fecha_min = pd.NaT
    fecha_max = pd.NaT

    if col_fecha is not None and col_fecha in df.columns:
        col = pd.to_datetime(df[col_fecha], errors='coerce')
        fecha_min = col.min()
        fecha_max = col.max()

    return {
        'dataset':            nombre,
        'filas':              len(df),
        'columnas':           len(df.columns),
        'nulos_pct_max':      round(df.isnull().mean().max() * 100, 2),
        'duplicados_exactos': int(df.duplicated().sum()),
        'fecha_min':          fecha_min,
        'fecha_max':          fecha_max,
    }

In [9]:
# Aplicamos la auditoría a los 12 datasets.
# col_fecha apunta a la columna de fecha operativa de cada dataset,
# no a metadatos del informe (report_start, report_end).
# Los datasets sin dimensión temporal propia reciben col_fecha=None.

auditoria = pd.DataFrame([
    audit_dataset('tickets',        df_tickets,       col_fecha='date'),
    audit_dataset('ventas',         df_ventas,        col_fecha='report_start'),
    audit_dataset('reservas',       df_reservas,      col_fecha='reservation_datetime'),
    audit_dataset('tips',           df_tips,          col_fecha=None),   # sin fecha propia: referencia document_id de tickets
    audit_dataset('articulos',      df_artic,         col_fecha=None),   # catálogo estático
    audit_dataset('departamentos',  df_depart,        col_fecha=None),   # catálogo estático
    audit_dataset('menu',           df_menu,          col_fecha=None),   # catálogo estático
    audit_dataset('festivos',       df_festivos,      col_fecha='fecha'),
    audit_dataset('eventos',        df_eventos,       col_fecha='fecha_inicio'),
    audit_dataset('meteo_diaria',   meteo_diaria,     col_fecha='date'),
    audit_dataset('meteo_horaria',  meteo_horaria,    col_fecha='datetime'),
    audit_dataset('total_articles', total_articles,   col_fecha=None),   # agregado global sin granularidad diaria
])

display(auditoria)

,dataset,filas,columnas,nulos_pct_max,duplicados_exactos,fecha_min,fecha_max
0,tickets,6709,11,0.00,0,2025-10-02 00:00:00,2026-07-09 00:00:00
1,ventas,5532,13,0.00,0,2025-10-01 00:00:00,2026-07-06 00:00:00
2,reservas,21341,20,99.99,0,2022-11-10 13:45:00,2026-07-25 14:00:00
3,tips,1697,11,0.00,0,NaT,NaT
4,articulos,405,5,95.56,0,NaT,NaT
5,departamentos,23,4,100.00,0,NaT,NaT
6,menu,405,3,0.00,0,NaT,NaT
7,festivos,24,4,0.00,0,2025-01-01 00:00:00,2026-12-25 00:00:00
8,eventos,78,20,0.00,0,2025-10-10 00:00:00,2026-09-16 00:00:00
9,meteo_diaria,921,9,0.00,0,2024-01-01 00:00:00,2026-07-09 00:00:00


### Conclusiones de la auditoría inicial

- **Sin duplicados en ningún dataset.** Los 12 datasets pasan el check de filas
  idénticas con resultado 0. No hay registros duplicados que eliminar en esta capa.

- **Los catálogos (artículos, departamentos) presentan nulos estructurales.**
  `article_short_name` está vacío en el 95.6% de los artículos y
  `department_short_name` al 100%. Ambas columnas son residuos del sistema TPV
  sin valor para el modelo; se eliminarán en la sección 2.

- **Reservas acumula el mayor problema de calidad.** El 99.99% de nulos en
  `nulos_pct_max` corresponde a la columna `reference`, que está prácticamente
  vacía. Adicionalmente, `entered_by` (85.6% nulos) y `referrer` (70.2% nulos)
  son columnas operativas del TPV sin utilidad para la predicción. Las columnas
  clave —`reservation_datetime`, `status`, `shift`, `people`, `origin`— tienen
  0 nulos. Los problemas específicos de reservas se detallan en la sección 2.3.

- **Tickets, ventas, meteorología, festivos y eventos están en buen estado
  general.** Sin nulos en columnas operativas y sin duplicados. Los problemas
  puntuales detectados (días sin actividad en tickets, una fila con importe
  negativo en ventas, outliers de importe) se abordan en la sección 2.